# Clothing Retrieval — CLIP Embedding Indexer

Constructs FAISS-based searchable indexes for **Config A** and **Config B** of the comparison study.

---

### Embedding formula
```
vec = beta * CLIP_vision(crop) + (1 - beta) * CLIP_language(caption)
```
Vectors are unit-normalised prior to storage. Inner-product search is equivalent to cosine similarity.

---

### Configurations produced here
| Setup | beta | Text used? | CLIP mode |
|--------|------|-----------|------------|
| A | 1.0 | No | Fixed |
| B | 0.7 | Yes | Fixed |
| B | 0.5 | Yes | Fixed |

---

### Source Data
- `vr-yolo-bbox-cropped-images` — bounding-box crops alongside master_crops.csv
- `blip-captions-data` — contains gallery_captions.json

### Produced Artifacts
- `gallery_vectors_A_b10.npy`, `gallery_vectors_B_b07.npy`, `gallery_vectors_B_b05.npy`
- `idx_A_b10.bin`, `idx_B_b07.bin`, `idx_B_b05.bin`  *(HNSW indexes)*
- `item_index_map.csv`

## Step 1 — Package Installation

In [1]:
!pip uninstall -y faiss faiss-gpu
!pip install ftfy regex transformers faiss-cpu Pillow --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.3 MB/s eta 0:00:00


## Step 2 — Library Imports

In [2]:
import os
import json
import numpy as np
import pandas as pd
import torch
import faiss
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
import warnings
warnings.filterwarnings('ignore')

GPU = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Runtime device  : {GPU}')
if GPU == 'cuda':
    print(f'GPU name        : {torch.cuda.get_device_name(0)}')
print(f'PyTorch version : {torch.__version__}')
print(f'FAISS version   : {faiss.__version__}')
print('Initialisation finished!')

Runtime device  : cuda
GPU name        : Tesla T4
PyTorch version : 2.10.0+cu128
FAISS version   : 1.13.2
Initialisation finished!


## Step 3 — Directory Paths and Settings

In [3]:
# ── Data source directories ──────────────────────────────────────────────────
BBOX_CROPS_DIR   = '/kaggle/input/datasets/akibatra25/vr-yolo-bbox-cropped-images'
CAPTIONS_DIR     = '/kaggle/input/datasets/akibatra25/blip-captions-data'

# ── Destination for produced files ──────────────────────────────────────────
OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Pre-trained model checkpoint ─────────────────────────────────────────────
CLIP_CKPT = 'openai/clip-vit-base-patch32'

# ── Blend weight values to process ──────────────────────────────────────────
# beta=1.0 → Config A: image embedding only
# beta=0.7 → Config B: 70% image, 30% text contribution
# beta=0.5 → Config B: equal image-text contribution
BETA_VALUES = [1.0, 0.7, 0.5]

BATCH_SZ = 64   # images processed per forward pass

for tag, p in [('BBOX_CROPS_DIR', BBOX_CROPS_DIR), ('CAPTIONS_DIR', CAPTIONS_DIR)]:
    status_msg = 'Found ✓' if os.path.exists(p) else 'NOT FOUND ✗'
    print(f'[{status_msg}] {tag}: {p}')

[Found ✓] BBOX_CROPS_DIR: /kaggle/input/datasets/akibatra25/vr-yolo-bbox-cropped-images
[Found ✓] CAPTIONS_DIR: /kaggle/input/datasets/akibatra25/blip-captions-data


## Step 4 — Read Gallery Metadata and Resolve Paths

In [4]:
master_df = pd.read_csv(os.path.join(BBOX_CROPS_DIR, 'master_crops.csv'))
gallery_df  = master_df[master_df['split'] == 'gallery'].reset_index(drop=True)

print(f'Total master rows : {len(master_df):,}')
print(f'Gallery rows      : {len(gallery_df):,}')

def translate_path(saved_path):
    if pd.isna(saved_path):
        return saved_path
    for pfx in ['/kaggle/working/', '/kaggle/input/']:
        if saved_path.startswith(pfx):
            tail = saved_path.replace(pfx, '')
            for ds_name in ['yolo-bbox-crops-v1/', 'datasets/harshitabansal307/yolo-bbox-crops-v1/']:
                tail = tail.replace(ds_name, '')
            return os.path.join(BBOX_CROPS_DIR, tail)
    return saved_path

gallery_df['img_path'] = gallery_df['crop_path'].apply(translate_path)
gallery_df['on_disk']  = gallery_df['img_path'].apply(
    lambda p: os.path.exists(p) if isinstance(p, str) else False
)

print(f'Gallery crops on disk: {gallery_df["on_disk"].sum():,} / {len(gallery_df):,}')

if gallery_df['on_disk'].sum() < len(gallery_df) * 0.9:
    print('Attempting direct file-path resolution...')
    def direct_path(img_name):
        rel = img_name[4:] if img_name.startswith('img/') else img_name
        for sub in ['data/bbox_crops', 'data/yolo_crops']:
            p = os.path.join(BBOX_CROPS_DIR, sub, rel)
            if os.path.exists(p): return p
        return os.path.join(BBOX_CROPS_DIR, 'data/bbox_crops', rel)
    gallery_df['img_path'] = gallery_df['image_name'].apply(direct_path)
    gallery_df['on_disk']  = gallery_df['img_path'].apply(os.path.exists)
    print(f'After direct path: {gallery_df["on_disk"].sum():,} / {len(gallery_df):,}')

print('\nSample paths:')
for p in gallery_df['img_path'].head(2):
    print(f'  [{"OK" if os.path.exists(p) else "MISSING"}] {p}')

Total master rows : 52,712
Gallery rows      : 12,612
Gallery crops on disk: 12,612 / 12,612

Sample paths:
  [OK] /kaggle/input/datasets/akibatra25/vr-yolo-bbox-cropped-images/data/bbox_crops/WOMEN/Blouses_Shirts/id_00000001/02_1_front.jpg
  [OK] /kaggle/input/datasets/akibatra25/vr-yolo-bbox-cropped-images/data/bbox_crops/WOMEN/Blouses_Shirts/id_00000001/02_3_back.jpg


In [5]:
with open(os.path.join(CAPTIONS_DIR, 'gallery_captions.json')) as fh:
    caption_dict = json.load(fh)

print(f'Gallery captions loaded : {len(caption_dict):,}')
gallery_names = set(gallery_df['image_name'].tolist())
matched_names   = gallery_names.intersection(set(caption_dict.keys()))
print(f'Captions covering gallery: {len(matched_names):,} / {len(gallery_names):,}')
print('\nSample captions:')
for nm, cap in list(caption_dict.items())[:3]:
    print(f'  {nm.split("/")[-1]}: {cap}')

Gallery captions loaded : 12,612
Captions covering gallery: 12,612 / 12,612

Sample captions:
  02_1_front.jpg: a woman in black shorts and a white blouse
  02_3_back.jpg: the back view of a woman wearing a white blouse
  01_1_front.jpg: a woman wearing a tank top with the words snoop dogg on it


## Step 5 — Initialise Frozen CLIP Model

In [6]:
print(f'Loading CLIP checkpoint: {CLIP_CKPT}')
clip_processor  = CLIPProcessor.from_pretrained(CLIP_CKPT)
clip_model   = CLIPModel.from_pretrained(CLIP_CKPT).to(GPU)

for p in clip_model.parameters():
    p.requires_grad = False
clip_model.eval()

EMBED_DIM = clip_model.config.projection_dim
print(f'CLIP ready — all weights frozen!')
print(f'Vector dimension: {EMBED_DIM}')

Loading CLIP checkpoint: openai/clip-vit-base-patch32


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP ready — all weights frozen!
Vector dimension: 512


## Step 6 — Define Embedding Utilities

In [7]:
def batch_encode_images(path_list):
    imgs, valid_indices = [], []
    for i, p in enumerate(path_list):
        try:
            imgs.append(Image.open(p).convert('RGB'))
            valid_indices.append(i)
        except Exception:
            pass
    if not imgs:
        return None, valid_indices
    inp = clip_processor(images=imgs, return_tensors='pt', padding=True).to(GPU)
    with torch.no_grad():
        raw = clip_model.get_image_features(**inp)
        vecs = raw.pooler_output if hasattr(raw, 'pooler_output') and not isinstance(raw, torch.Tensor) else raw
    vecs = vecs / vecs.norm(dim=-1, keepdim=True)
    return vecs.cpu().numpy(), valid_indices


def batch_encode_texts(text_list):
    inp = clip_processor(text=text_list, return_tensors='pt', padding=True, truncation=True, max_length=77).to(GPU)
    with torch.no_grad():
        raw = clip_model.get_text_features(**inp)
        vecs = raw.pooler_output if hasattr(raw, 'pooler_output') and not isinstance(raw, torch.Tensor) else raw
    vecs = vecs / vecs.norm(dim=-1, keepdim=True)
    return vecs.cpu().numpy()


def blend_vectors(img_vecs, txt_vecs, beta):
    blended = beta * img_vecs + (1.0 - beta) * txt_vecs
    norms   = np.linalg.norm(blended, axis=-1, keepdims=True)
    return blended / np.maximum(norms, 1e-8)


print('Embedding helpers initialised ✓')
# Quick test
_row = gallery_df[gallery_df['on_disk']].iloc[0]
_ie, _ = batch_encode_images([_row['img_path']])
_te     = batch_encode_texts(['a blue dress'])
print(f'Image vec shape : {_ie.shape}  norm: {np.linalg.norm(_ie[0]):.4f}')
print(f'Text vec shape  : {_te.shape}  norm: {np.linalg.norm(_te[0]):.4f}')

Embedding helpers initialised ✓
Image vec shape : (1, 512)  norm: 1.0000
Text vec shape  : (1, 512)  norm: 1.0000


## Step 7 — Generate Gallery Image Embeddings

In [8]:
active_gallery = gallery_df[gallery_df['on_disk']].reset_index(drop=True)
N = len(active_gallery)
print(f'Encoding {N:,} gallery images...')

vis_embs = np.zeros((N, EMBED_DIM), dtype=np.float32)
failed_positions    = []

for s in tqdm(range(0, N, BATCH_SZ), desc='Vision encoding'):
    chunk = active_gallery.iloc[s : s + BATCH_SZ]
    vecs, status_msg = batch_encode_images(chunk['img_path'].tolist())
    if vecs is None:
        failed_positions.extend(range(s, s + len(chunk)))
        continue
    for li, gi in enumerate(status_msg):
        vis_embs[s + gi] = vecs[li]

print(f'Image encoding finished — shape: {vis_embs.shape}, failed: {len(failed_positions)}')

Encoding 12,612 gallery images...


Vision encoding: 100%|██████████| 198/198 [02:05<00:00,  1.58it/s]

Image encoding finished — shape: (12612, 512), failed: 0


## Step 8 — Generate Gallery Caption Embeddings

In [9]:
print(f'Encoding {N:,} gallery captions...')
lang_embs    = np.zeros((N, EMBED_DIM), dtype=np.float32)
fallback_count    = 0

for s in tqdm(range(0, N, BATCH_SZ), desc='Language encoding'):
    chunk = active_gallery.iloc[s : s + BATCH_SZ]
    caps  = []
    for _, r in chunk.iterrows():
        c = caption_dict.get(r['image_name'], '')
        if not c:
            c = 'a clothing item'
            fallback_count += 1
        caps.append(c)
    vecs = batch_encode_texts(caps)
    lang_embs[s : s + len(caps)] = vecs

print(f'Caption encoding finished — shape: {lang_embs.shape}')
print(f'Default captions substituted: {fallback_count}')

Encoding 12,612 gallery captions...


Language encoding: 100%|██████████| 198/198 [00:07<00:00, 26.61it/s]

Caption encoding finished — shape: (12612, 512)
Default captions substituted: 0


## Step 9 — Construct FAISS Index per Beta Setting

In [10]:
for beta in BETA_VALUES:
    print(f'\n--- Beta = {beta} ---')

    if beta == 1.0:
        config_label  = 'A'
        beta_label = 'b10'
        combined_vec    = vis_embs.copy()
        print('Config A: image features only (text disabled)')
    elif beta == 0.7:
        config_label  = 'B'
        beta_label = 'b07'
        combined_vec    = blend_vectors(vis_embs, lang_embs, beta)
        print(f'Config B: {beta*100:.0f}% vision + {(1-beta)*100:.0f}% language')
    else:
        config_label  = 'B'
        beta_label = f'b{int(beta*10):02d}'
        combined_vec    = blend_vectors(vis_embs, lang_embs, beta)
        print(f'Config B: {beta*100:.0f}% vision + {(1-beta)*100:.0f}% language')

    print(f'Vector matrix shape : {combined_vec.shape}')
    print(f'Sample L2 norm      : {np.linalg.norm(combined_vec[0]):.4f}  (expected ~1.0)')

    # Persist the embedding matrix
    vector_path = os.path.join(OUT_DIR, f'gallery_vectors_{config_label}_{beta_label}.npy')
    np.save(vector_path, combined_vec)
    print(f'Vectors saved → {vector_path}')

    # Construct FAISS HNSW index for approximate nearest-neighbour search
    # Inner-product space is used; unit-normalised vectors make IP equal to cosine similarity
    M         = 32   # edges per node — governs graph density
    ef_search = 64   # query-time beam width — larger values improve accuracy at cost of speed
    ef_constr = 200  # build-time beam width — larger values yield a higher-quality graph

    # Assemble the HNSW graph structure
    hnsw_index  = faiss.IndexHNSWFlat(EMBED_DIM, M, faiss.METRIC_INNER_PRODUCT)
    hnsw_index.hnsw.efSearch       = ef_search
    hnsw_index.hnsw.efConstruction = ef_constr
    hnsw_index.add(combined_vec.astype(np.float32))
    print(f'HNSW index: {hnsw_index.ntotal:,} vectors  (M={M}, efSearch={ef_search}, efConstruction={ef_constr})')

    index_path = os.path.join(OUT_DIR, f'idx_{config_label}_{beta_label}.bin')
    faiss.write_index(hnsw_index, index_path)
    print(f'Index saved → {index_path}')

print('\nAll FAISS indexes constructed!')



--- Beta = 1.0 ---
Config A: image features only (text disabled)
Vector matrix shape : (12612, 512)
Sample L2 norm      : 1.0000  (expected ~1.0)
Vectors saved → /kaggle/working/gallery_vectors_A_b10.npy
HNSW index: 12,612 vectors  (M=32, efSearch=64, efConstruction=200)
Index saved → /kaggle/working/idx_A_b10.bin

--- Beta = 0.7 ---
Config B: 70% vision + 30% language
Vector matrix shape : (12612, 512)
Sample L2 norm      : 1.0000  (expected ~1.0)
Vectors saved → /kaggle/working/gallery_vectors_B_b07.npy
HNSW index: 12,612 vectors  (M=32, efSearch=64, efConstruction=200)
Index saved → /kaggle/working/idx_B_b07.bin

--- Beta = 0.5 ---
Config B: 50% vision + 50% language
Vector matrix shape : (12612, 512)
Sample L2 norm      : 1.0000  (expected ~1.0)
Vectors saved → /kaggle/working/gallery_vectors_B_b05.npy
HNSW index: 12,612 vectors  (M=32, efSearch=64, efConstruction=200)
Index saved → /kaggle/working/idx_B_b05.bin

All FAISS indexes constructed!


## Step 10 — Persist the Item-to-Index Lookup Table

In [11]:
pos_map = active_gallery[['image_name', 'item_id', 'split', 'clothes_type', 'img_path']].copy()
pos_map = pos_map.rename(columns={'img_path': 'crop_path'})
pos_map['faiss_index_pos'] = range(len(pos_map))

map_path = os.path.join(OUT_DIR, 'item_index_map.csv')
pos_map.to_csv(map_path, index=False)
print(f'Item index map saved → {map_path}')
print(f'Rows    : {len(pos_map):,}')
print(f'Columns : {pos_map.columns.tolist()}')

Item index map saved → /kaggle/working/item_index_map.csv
Rows    : 12,612
Columns : ['image_name', 'item_id', 'split', 'clothes_type', 'crop_path', 'faiss_index_pos']


## Step 11 — Verification Check

In [12]:
import matplotlib.pyplot as plt

verify_idx = faiss.read_index(os.path.join(OUT_DIR, 'idx_A_b10.bin'))
verify_vecs  = np.load(os.path.join(OUT_DIR, 'gallery_vectors_A_b10.npy'))

sample_row = active_gallery.sample(1, random_state=42).iloc[0]
sample_pos = sample_row.name
sample_vec = verify_vecs[sample_pos:sample_pos+1].astype(np.float32)

scores, neighbors = verify_idx.search(sample_vec, 6)

print(f'Query : {sample_row["image_name"].split("/")[-1]}  (item_id={sample_row["item_id"]})')
print()
print('Top 6 retrieved:')
for rank, (h, d) in enumerate(zip(neighbors[0], scores[0])):
    r = pos_map.iloc[h]
    tag = '✓ MATCH' if r['item_id'] == sample_row['item_id'] else '✗'
    print(f'  Rank {rank+1}: item_id={r["item_id"]}  score={d:.4f}  {tag}')

Query : 04_1_front.jpg  (item_id=id_00004024)

Top 6 retrieved:
  Rank 1: item_id=id_00004024  score=1.0000  ✓ MATCH
  Rank 2: item_id=id_00002056  score=0.9572  ✗
  Rank 3: item_id=id_00002226  score=0.9494  ✗
  Rank 4: item_id=id_00007769  score=0.9492  ✗
  Rank 5: item_id=id_00007846  score=0.9463  ✗
  Rank 6: item_id=id_00004551  score=0.9456  ✗


## Step 12 — Run Summary

In [13]:
print('=' * 60)
print('    CLIP INDEXING COMPLETE')
print('=' * 60)
print(f'  CLIP checkpoint     : {CLIP_CKPT}')
print(f'  Gallery vectors     : {len(active_gallery):,}')
print(f'  Vector dimension    : {EMBED_DIM}')
print()
print('  Output files:')
for fn in sorted(os.listdir(OUT_DIR)):
    if fn.endswith(('.bin', '.npy', '.csv')):
        sz = os.path.getsize(os.path.join(OUT_DIR, fn)) / 1e6
        print(f'    {fn}  ({sz:.1f} MB)')
print()
print('  Next steps:')
print('    1. Save output as Kaggle dataset (e.g. clip-indexes-ab-friend)')
print('    2. Run evaluation notebook')
print('    3. Run CLIP fine-tuning notebook for Config C')
print('=' * 60)

    CLIP INDEXING COMPLETE
  CLIP checkpoint     : openai/clip-vit-base-patch32
  Gallery vectors     : 12,612
  Vector dimension    : 512

  Output files:
    gallery_vectors_A_b10.npy  (25.8 MB)
    gallery_vectors_B_b05.npy  (25.8 MB)
    gallery_vectors_B_b07.npy  (25.8 MB)
    idx_A_b10.bin  (29.3 MB)
    idx_B_b05.bin  (29.3 MB)
    idx_B_b07.bin  (29.3 MB)
    item_index_map.csv  (2.5 MB)

  Next steps:
    1. Save output as Kaggle dataset (e.g. clip-indexes-ab-friend)
    2. Run evaluation notebook
    3. Run CLIP fine-tuning notebook for Config C
